## Eval models

In [1]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd

In [2]:
df = pd.read_csv("../data/eval_testset_gpt-5.4.csv")

In [3]:
df

,speech_id,debate_id,amendment_author_name,amendment_author_group,amendment_content,amendment_summary,speech_date,speaker_name,speaker_group,speaker_government,label,speech,predicted_label,position_raw,confiance,raison,parse_error
0,VTANR5L15V2035-15-754697,VTANR5L15V2035,Jean-Luc Mélenchon,FI,Compléter cet article par les trois alinéas su...,Cet amendement a pour objectif que les algorit...,2019-07-04,Cédric O,LREM,True,CONTRE,Parmi les thèmes que le Gouvernement souhaite ...,1,POUR,haute,Le député reconnaît explicitement le problème ...,False
1,VTANR5L15V423-15-669761,VTANR5L15V423,Bastien Lachaud,FI,"À l’alinéa 103, supprimer les mots : « ou de c...",Le présent amendement vise à exclure la « cont...,2018-03-21,Jean-Luc Mélenchon,LFI,False,POUR,Nous venons d’entendre Mme la ministre et nous...,1,POUR,haute,Le député critique explicitement la « guerre c...,False
2,VTANR5L15V1137-15-845948,VTANR5L15V1137,Loïc Prud'homme,FI,"Compléter l’alinéa 7 par les mots : « , en pri...","Selon nous, la question à laquelle doit répond...",2018-09-14,Jean-Luc Mélenchon,LFI,False,POUR,"J’avoue être assez stupéfait, madame Pompili, ...",1,POUR,haute,Jean-Luc Mélenchon défend clairement l’amendem...,False
3,VTANR5L15V1208-15-778472,VTANR5L15V1208,Daniel Fasquelle,LR,Supprimer cet article.,L’article 71 habilite le gouvernement à légifé...,2018-10-05,Coralie Dubost,LREM,False,CONTRE,L’article 71 ter est issu d’un amendement dé...,0,CONTRE,haute,Le député indique explicitement qu’il donne « ...,False
4,VTANR5L15V1199-15-202657,VTANR5L15V1199,Dominique Potier,SOC,Rédiger ainsi cet article : « I. – L’article L...,"Depuis une trentaine d’années, le mouvement de...",2018-10-05,Coralie Dubost,LREM,False,CONTRE,Nous sommes particulièrement fiers de renforce...,0,CONTRE,haute,La députée dit explicitement que les amendemen...,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1193,VTANR5L17V5542-17-62989,VTANR5L17V5542,Patrick Hetzel,DR,Compléter cet article par les deux alinéas sui...,L’aide à mourir est incompatible avec les disp...,2026-02-19,Brigitte Liso,LREM,False,CONTRE,Ces amendements sont superfétatoires car il n’...,0,CONTRE,haute,La députée indique que les amendements sont « ...,False
1194,VTANR5L17V1852-17-72976,VTANR5L17V1852,Justine Gruet,DR,Rédiger ainsi l’alinéa 9 : « 5° Exprimer son c...,Cet amendement vise à s’assurer que le consent...,2025-05-19,Justine Gruet,LR,False,POUR,Afin de sécuriser le consentement libre et écl...,1,POUR,haute,La députée explique qu’il faut mieux sécuriser...,False
1195,VTANR5L17V956-17-165823,VTANR5L17V956,Géraldine Grangier,RN,"I. – À l’alinéa 2, après le mot : « peut », i...",Cet amendement vise à clarifier l'exercice du ...,2025-03-11,Peio Dufau,SOC,False,CONTRE,Défavorable.,0,CONTRE,haute,Le député indique explicitement « Défavorable ...,False
1196,VTANR5L17V1043-17-205637,VTANR5L17V1043,Ugo Bernalicis,LFI-NFP,"I. – Après l’alinéa 16, insérer l’alinéa suiva...","Par cet amendement d'appel, les député.es du g...",2025-03-18,Vincent Caure,LREM,False,CONTRE,Vous vous inquiétez de la répartition des comp...,0,CONTRE,haute,Le député conclut explicitement par « Avis déf...,False


In [7]:

def compute_eval_metrics(df, model_prompt):
    # 1. Map gold labels to 0/1
    df["gold_label"] = df["label"].map({"POUR": 1, "CONTRE": 0})

    # 2. Filter out invalid predictions (-1)
    valid = df[df["predicted_label"] != -1]

    y_true = valid["gold_label"]
    y_pred = valid["predicted_label"]

    # 3. Compute metrics
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)    

    # --- Error breakdown ---
    errors = (y_true != y_pred)
    n_errors = errors.sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    fp_pct = fp / n_errors if n_errors > 0 else 0
    fn_pct = fn / n_errors if n_errors > 0 else 0

    # --- Class-wise F1 ---
    f1_pos = f1_score(y_true, y_pred, pos_label=1)
    f1_neg = f1_score(y_true, y_pred, pos_label=0)

    
    # Return as a DataFrame row
    return pd.DataFrame([{
        "model_prompt": model_prompt,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "accuracy": accuracy,
        "f1_pour(1)": f1_pos,
        "f1_contre(0)": f1_neg,
        "fp_pct_errors": fp_pct,
        "fn_pct_errors": fn_pct,
        "n_errors": n_errors,
        "n_samples": len(valid),
        "n_total": len(df)
    }])


In [8]:
# Load your annotated dataframe
#models_prompts = ["llama_basic", "llama_guide_context", "llama_guide", "llama_llama_context", "llama_llama", "qwen_guide_context", "qwen_guide", "qwen_llama_context", "qwen_llama", "mistral_guide", "mistral_llama"] 
models_prompts = ["gpt-5.4"]
results = []
for model_prompt in models_prompts:
    print("\n ======================= \n")
    print(f"COMPUTING EVALS METRICS FOR MODEL/PROMPT {model_prompt}")
    df = pd.read_csv('../data/eval_testset_' + model_prompt + '.csv', low_memory=False)
    res = compute_eval_metrics(df, model_prompt)
    results.append(res)

# Concatenate into one results table
results_df = pd.concat(results, ignore_index=True)





COMPUTING EVALS METRICS FOR MODEL/PROMPT gpt-5.4


In [9]:

results_df


,model_prompt,precision,recall,f1_score,accuracy,f1_pour(1),f1_contre(0),fp_pct_errors,fn_pct_errors,n_errors,n_samples,n_total
0,gpt-5.4,0.847682,0.866328,0.856904,0.857262,0.856904,0.857619,0.538012,0.461988,171,1198,1198


In [10]:
errors = df[df["gold_label"] != df["predicted_label"]]

In [11]:
errors

,speech_id,debate_id,amendment_author_name,amendment_author_group,amendment_content,amendment_summary,speech_date,speaker_name,speaker_group,speaker_government,label,speech,predicted_label,position_raw,confiance,raison,parse_error,gold_label
0,VTANR5L15V2035-15-754697,VTANR5L15V2035,Jean-Luc Mélenchon,FI,Compléter cet article par les trois alinéas su...,Cet amendement a pour objectif que les algorit...,2019-07-04,Cédric O,LREM,True,CONTRE,Parmi les thèmes que le Gouvernement souhaite ...,1,POUR,haute,Le député reconnaît explicitement le problème ...,False,0
14,VTANR5L15V1029-15-524677,VTANR5L15V1029,Éric Coquerel,FI,Après la première phrase du premier alinéa de ...,"Par cet amendement, nous proposons de consacre...",2018-07-12,Éric Coquerel,LFI,False,POUR,"Mais, au-delà de cet amendement, ce qui m’inqu...",0,CONTRE,moyenne,Le député exprime une inquiétude sur le conten...,False,1
25,VTANR5L15V1802-15-49943,VTANR5L15V1802,Delphine Bagarry,LAREM,"À l'alinéa 8, après le mot : « réadaptation, »...","Cet amendement donne la possibilité, en foncti...",2019-03-22,Brahim Hammouche,MODEM,False,POUR,Je comprends le sens de l’amendement et des po...,0,CONTRE,haute,"Le député dit comprendre l’amendement, mais es...",False,1
29,VTANR5L15V1951-15-584431,VTANR5L15V1951,Hubert Wulfranc,GDR,La dernière phrase du second alinéa du V des a...,Les auteurs de l’amendement préconisent un rac...,2019-06-14,Élisabeth Borne,LREM,True,CONTRE,"Monsieur Bernalicis, je vous informe que la gr...",1,POUR,moyenne,L’intervention annonce la mise en œuvre de la ...,False,0
33,VTANR5L15V2107-15-595735,VTANR5L15V2107,Coralie Dubost,LAREM,"Compléter l’alinéa 26 par les mots : « , à la ...",Cet amendement a pour objet de faciliter le re...,2019-10-02,Agnès Buzyn,LREM,True,CONTRE,La véritable limite n’est pas matérielle : ell...,1,POUR,moyenne,Agnès Buzyn souligne que la loi doit permettre...,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1160,VTANR5L17V736-17-236136,VTANR5L17V736,Davy Rimane,GDR,Compléter le titre par les mots : « nonobstant...,Cet amendement vise à reformuler le titre de l...,2025-02-06,Bastien Lachaud,LFI,False,POUR,Mme Le Pen a été très claire. Le texte que nou...,0,CONTRE,haute,L’intervention critique le texte en le présent...,False,1
1166,VTANR5L17V1448-17-42503,VTANR5L17V1448,Lisa Belluco,ECOS,Supprimer les alinéas 25 à 31.,L'objet de ces alinéas est de réduire la trans...,2025-04-11,Laurent Marcangeli,HOR,True,POUR,Vous voyez qu’il est toujours intéressant de d...,0,CONTRE,haute,Le député donne un « avis favorable » aux alin...,False,1
1178,VTANR5L17V1700-17-292704,VTANR5L17V1700,Sébastien Huyghe,EPR,I. – Lorsqu’un projet d’exploitation de carriè...,Le présent amendement vise à simplifier la réa...,2025-05-15,Valérie Létard,LIOT,True,POUR,"Même avis, par cohérence avec le débat que nou...",0,CONTRE,haute,Le député dit qu’il faut « limiter les dérogat...,False,1
1182,VTANR5L17V1405-17-21054,VTANR5L17V1405,Matthias Renault,RN,Le chapitre II du titre II du livre III du cod...,Dans le triple objectif de dégager des économi...,2025-04-11,Jean-Luc Fugit,LREM,False,CONTRE,"Je reprends la parole, même si je suis déjà in...",1,POUR,moyenne,L’intervention insiste sur l’importance du suj...,False,0
